# Day 4

## コードでトークン化

In [2]:
import tiktoken

encoding = tiktoken.encoding_for_model("gpt-4.1-mini")

tokens = encoding.encode("Hi my name is Ed and I like banoffee pie")

In [3]:
tokens

[12194, 922, 1308, 382, 6117, 326, 357, 1299, 9171, 26458, 5148]

In [4]:
for token_id in tokens:
    token_text = encoding.decode([token_id])
    print(f"{token_id} = {token_text}")

12194 = Hi
922 =  my
1308 =  name
382 =  is
6117 =  Ed
326 =  and
357 =  I
1299 =  like
9171 =  ban
26458 = offee
5148 =  pie


In [5]:
encoding.decode([326])

' and'

# もう一つのトピック！

### 「記憶」という幻想

すでにご存知の方も多いと思いますが、知らない方には「あっ、そういうことか！」という気づきになるかもしれません。

In [6]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not api_key.startswith("sk-proj-"):
    print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook")
else:
    print("API key found and looks good so far!")

API key found and looks good so far!


### 次のセルが何をしているか、しっかり理解できているはずです！

_OpenAI Python クライアントライブラリの新しいインスタンスを作っています。これは、GPT LLM（またはその他の LLM プロバイダー）を呼び出すエンドポイントへの HTTP リクエストを薄くラップしたものです。_

In [7]:
from openai import OpenAI

openai = OpenAI()

### OpenAI へのメッセージは辞書のリスト

In [8]:
messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": "Hi! I'm Ed!"}
    ]

In [9]:
response = openai.chat.completions.create(model="gpt-4.1-mini", messages=messages)
response.choices[0].message.content

'Hello Ed! How can I assist you today?'

### では、フォローアップの質問をしてみましょう

In [10]:
messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": "What's my name?"}
    ]

In [11]:
response = openai.chat.completions.create(model="gpt-4.1-mini", messages=messages)
response.choices[0].message.content

'I’m not sure what your name is based on our current conversation. Could you please tell me your name?'

### え、ちょっと待って？？

さっき教えたじゃないですか！

何が起きているのでしょう？

ポイントはここです：LLM へのすべての呼び出しは完全に**ステートレス**です。毎回、完全に新しい呼び出しになります。AI エンジニアとして、LLM に「記憶」があるように見せる技術を考案するのは**私たちの仕事**です。

In [12]:
messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": "Hi! I'm Ed!"},
    {"role": "assistant", "content": "Hi Ed! How can I assist you today?"},
    {"role": "user", "content": "What's my name?"}
    ]

In [13]:
response = openai.chat.completions.create(model="gpt-4.1-mini", messages=messages)
response.choices[0].message.content

'Your name is Ed! How can I help you today, Ed?'

## まとめ

当たり前に思えるかもしれませんが、改めて確認しましょう：

1. LLM へのすべての呼び出しはステートレスである
2. 毎回、入力プロンプトにその時点までの会話全体を渡している
3. これにより、LLM が会話の文脈を保持しているように見える「記憶の幻想」が生まれる
4. しかしこれはトリックにすぎない。毎回会話全体を渡した副産物である
5. LLM はシーケンスの次のトークンを予測するだけ。そのシーケンスに「私の名前は Ed です」と後から「私の名前は？」があれば… Ed と予測する！

ChatGPT もまさにこのトリックを使っています。メッセージを送るたびに、会話全体が渡されています。

**「ということは、これまでの会話分も毎回余分に料金がかかるのでは？」**

その通りです。そしてそれが**望ましい**ことです。LLM にシーケンス全体を振り返りながら次のトークンを予測してほしいのですから、その計算をしてもらう必要があり、電気代を払う必要があります！
